# Day 7 — Custom Expectations and Capstone

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/great-expectations-certified/notebooks/day-07-custom-expectations-capstone.ipynb)

**Badge:** Exam  
**Course:** Great Expectations for Data Quality

---

## What you will learn

- The GX Custom Expectation class hierarchy: `Expectation` → `ColumnMapExpectation` → your subclass
- Required methods: `_validate` (logic) and `map_metric` (the metric key)
- How `_prescriptive_renderer` makes your expectation human-readable in Data Docs
- How to test a custom expectation against sample data with valid and invalid rows
- **Capstone:** a full four-step pipeline with two named suites, per-stage Checkpoints, Data Docs, and a CI gate

> **Tip:** The most common mistake is implementing `_validate` correctly but forgetting `_prescriptive_renderer` — your expectation shows as raw JSON in Data Docs. Always test by running a Checkpoint with `UpdateDataDocsAction` and checking the HTML report.

## Reading: Custom Expectations Guide

Official guide: [Custom Expectations](https://docs.greatexpectations.io/docs/core/define_expectations/custom_expectations/)

The class hierarchy for a column-map expectation:

```
Expectation
  └── ColumnMapExpectation          # validates each row independently
        └── YourCustomExpectation   # your subclass
```

**Required attributes and methods:**

| Member | Purpose |
|---|---|
| `map_metric` | String key naming the metric this expectation uses (e.g., `'column_values.match_email'`) |
| `_validate(...)` | Core logic: returns dict with `success` (bool) and `result` (dict) |
| `success_keys` | Tuple of kwargs that affect pass/fail (used for caching) |
| `_prescriptive_renderer` | Classmethod that returns a human-readable string for Data Docs |

**How GX evaluates a ColumnMapExpectation:**
1. GX calls `_validate` with the column values and any kwargs.
2. `_validate` applies a boolean mask (rows that pass vs. fail).
3. GX computes `unexpected_count`, `unexpected_list`, and `unexpected_percent` from that mask.
4. The expectation passes if `unexpected_count == 0` (or within `mostly` threshold).

## Install dependencies

In [ ]:
%pip install great_expectations --quiet

## Setup

In [ ]:
import great_expectations as gx
import pandas as pd
import numpy as np
import re
import sys
from typing import Optional
from datetime import datetime, timedelta

print('GX version:', gx.__version__)

## 1. Implementing `ExpectColumnValuesToBeValidEmail`

We subclass `ColumnMapExpectation`. The key contract:
- `map_metric` must be a unique dotted string naming the metric.
- `_validate` receives a `Batch`, column name, and any `success_keys`.
- It returns `{'success': bool, 'result': {...}}`.
- `_prescriptive_renderer` must return a plain-English string — this is what appears in Data Docs instead of raw JSON.

In [ ]:
# Email regex: RFC-5322-compatible simplified pattern
_EMAIL_REGEX = re.compile(r'^[\w.!#$%&\'*+/=?^_`{|}~-]+@[\w-]+(?:\.[\w-]+)+$')


class ExpectColumnValuesToBeValidEmail(gx.expectations.ColumnMapExpectation):
    """Expect all values in a column to be valid email addresses.

    A value is valid if it matches the pattern:
    local-part@domain.tld

    Kwargs:
        column (str): The column to validate.
        mostly (float): Proportion of rows that must pass (default 1.0).
    """

    # Unique metric key for this expectation type
    map_metric = 'column_values.match_valid_email'

    # kwargs that influence pass/fail outcome (used for result caching)
    success_keys = ('mostly',)

    # Default kwargs
    default_kwarg_values = {'mostly': 1.0}

    def _validate(
        self,
        column: pd.Series,
        mostly: float = 1.0,
        **kwargs,
    ) -> dict:
        """Apply email regex to each row; return pass/fail and unexpected list."""
        # Boolean mask: True where value IS a valid email
        valid_mask = column.astype(str).map(lambda v: bool(_EMAIL_REGEX.match(v)))

        unexpected_values = column[~valid_mask].tolist()
        unexpected_count  = len(unexpected_values)
        total_count       = len(column)
        unexpected_pct    = (unexpected_count / total_count * 100) if total_count > 0 else 0.0

        # 'mostly' threshold: pass if fraction of valid rows >= mostly
        success = (1.0 - unexpected_count / total_count) >= mostly if total_count > 0 else True

        return {
            'success': success,
            'result': {
                'unexpected_count':   unexpected_count,
                'unexpected_percent': round(unexpected_pct, 2),
                'unexpected_list':    unexpected_values[:20],  # cap at 20 for display
                'element_count':      total_count,
            },
        }

    @classmethod
    def _prescriptive_renderer(cls, configuration=None, **kwargs) -> str:
        """Return a human-readable description for Data Docs."""
        column = (
            configuration.kwargs.get('column', 'unknown column')
            if configuration else 'unknown column'
        )
        mostly = (
            configuration.kwargs.get('mostly', 1.0)
            if configuration else 1.0
        )
        if mostly < 1.0:
            return (
                f'At least {mostly * 100:.0f}% of values in column "{column}" '
                f'must be valid email addresses (local-part@domain.tld).'
            )
        return (
            f'All values in column "{column}" must be valid email addresses '
            f'(local-part@domain.tld).'
        )


print('ExpectColumnValuesToBeValidEmail defined')
print('map_metric:', ExpectColumnValuesToBeValidEmail.map_metric)
print('Prescriptive render (mock):',
    ExpectColumnValuesToBeValidEmail._prescriptive_renderer())

## 2. Testing the Custom Expectation on Sample Data

We build a DataFrame with a mix of valid and invalid email addresses and confirm:
- `success` is `False` (some rows are invalid).
- `unexpected_list` contains exactly the bad email values.

In [ ]:
# Sample DataFrame with known good and bad emails
email_df = pd.DataFrame({
    'user_id': range(1, 11),
    'email': [
        'alice@example.com',      # valid
        'bob.smith@corp.org',     # valid
        'carol+tag@sub.domain.io',# valid
        'dave@example.co.uk',     # valid
        'eve@',                   # INVALID — no domain
        'frank.com',              # INVALID — no @
        '@nolocalpart.com',       # INVALID — no local part
        'grace@valid.net',        # valid
        'not-an-email',           # INVALID
        'heidi@good.org',         # valid
    ]
})

INVALID_EMAILS = {'eve@', 'frank.com', '@nolocalpart.com', 'not-an-email'}
print('Email DataFrame:')
print(email_df.to_string())

In [ ]:
# Run the custom expectation using the lower-level _validate API
email_exp = ExpectColumnValuesToBeValidEmail(column='email')
result_dict = email_exp._validate(column=email_df['email'])

print('success:', result_dict['success'])
print('unexpected_count:', result_dict['result']['unexpected_count'])
print('unexpected_list:', result_dict['result']['unexpected_list'])

# Verify that unexpected_list matches exactly the known bad emails
unexpected_set = set(result_dict['result']['unexpected_list'])
assert unexpected_set == INVALID_EMAILS, (
    f'Expected {INVALID_EMAILS}, got {unexpected_set}'
)
print()
print('PASS: unexpected_list contains exactly the known bad emails')

In [ ]:
# Test mostly threshold: allow up to 50% invalids
email_exp_mostly = ExpectColumnValuesToBeValidEmail(column='email', mostly=0.5)
result_mostly = email_exp_mostly._validate(
    column=email_df['email'], mostly=0.5
)
# 6 valid / 10 total = 60% valid >= 50% threshold => should pass
print('mostly=0.5 test — success:', result_mostly['success'])
assert result_mostly['success'] is True, 'Expected success=True with mostly=0.5'
print('PASS: mostly threshold works correctly')

## 3. Wiring the Custom Expectation into a Suite

Custom expectations are used the same way as built-in ones: instantiate and call `suite.add_expectation()`.

In [ ]:
context = gx.get_context()

# Suite using the custom expectation
suite_users = context.suites.add(gx.ExpectationSuite(name='users.quality'))
suite_users.add_expectation(ExpectColumnValuesToBeValidEmail(column='email'))
suite_users.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column='email')
)
suite_users.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(column='user_id')
)
context.suites.save(suite_users)

print('users.quality suite:', len(suite_users.expectations), 'expectations')

# Validate clean email data
clean_emails = pd.DataFrame({
    'user_id': range(1, 6),
    'email': [
        'alice@example.com',
        'bob@corp.org',
        'carol@domain.io',
        'dave@example.co.uk',
        'grace@valid.net',
    ]
})

ds_users   = context.data_sources.add_pandas('users_source')
asset_users = ds_users.add_dataframe_asset('users_asset')
bd_users   = asset_users.add_batch_definition_whole_dataframe('users_batch')

vd_users = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='vd_users_quality',
        data=bd_users,
        suite=context.suites.get('users.quality'),
    )
)

result_users = vd_users.run(batch_parameters={'dataframe': clean_emails})
print('Validation on clean emails — success:', result_users.success)

## 4. Data Docs and `_prescriptive_renderer`

Without `_prescriptive_renderer`, Data Docs shows raw JSON like:
```json
{"type": "expect_column_values_to_be_valid_email", "kwargs": {"column": "email"}}
```

With it, Data Docs shows:
> "All values in column 'email' must be valid email addresses (local-part@domain.tld)."

We demonstrate the renderer output below, and show how it would look in an `UpdateDataDocsAction` run.

In [ ]:
# Demonstrate prescriptive renderer output
from great_expectations.core import ExpectationConfiguration

# Simulate how Data Docs calls the renderer
mock_config = ExpectationConfiguration(
    type='expect_column_values_to_be_valid_email',
    kwargs={'column': 'email', 'mostly': 0.99},
)

rendered = ExpectColumnValuesToBeValidEmail._prescriptive_renderer(
    configuration=mock_config
)
print('Data Docs would display:')
print(f'  "{rendered}"')

# Without prescriptive_renderer, it would show raw JSON:
print()
print('Without _prescriptive_renderer, Data Docs would show raw JSON:')
print('  {"type": "expect_column_values_to_be_valid_email", "kwargs": {"column": "email"}}')

In [ ]:
# Wire UpdateDataDocsAction so Data Docs is generated when checkpoint runs
update_docs = gx.checkpoint.UpdateDataDocsAction(name='update_data_docs')

cp_users = context.checkpoints.add(
    gx.Checkpoint(
        name='users_checkpoint',
        validation_definitions=[vd_users],
        actions=[update_docs],
        result_format={'result_format': 'COMPLETE'},
    )
)

cp_result = cp_users.run(batch_parameters={'dataframe': clean_emails})
print('users_checkpoint run complete — success:', cp_result.success)
print()
print('UpdateDataDocsAction fired — HTML report generated.')
print('In a FileDataContext: great_expectations/uncommitted/data_docs/local_site/')
print('Open with: context.open_data_docs()')

---

## Capstone: Full Four-Step Pipeline with CI Gate

This is the full capstone project for the course:

> A full GX validation layer added to a four-step data pipeline (ingest → clean → aggregate → load) using a multi-suite Checkpoint, auto-generated Data Docs served locally, and a CI gate script that exits non-zero on any validation failure.

**Architecture:**
- Two named suites: `pipeline.schema` (structure checks) and `pipeline.values` (content/business-rule checks)
- Per-stage Checkpoints that validate inputs before transforming
- A final multi-suite Checkpoint with `UpdateDataDocsAction`
- A `ci_gate()` function that exits 1 on any failure
- Proof that the gate fires when a bad row is injected

In [ ]:
# ============================================================
# CAPSTONE — full pipeline with GX validation layer
# ============================================================

import great_expectations as gx
import pandas as pd
import numpy as np
import sys, re
from datetime import datetime, timedelta

# Fresh context for the capstone
ctx = gx.get_context()

# ---- Synthetic pipeline data ----
rng = np.random.default_rng(99)
N   = 400

raw_pipeline_df = pd.DataFrame({
    'event_id':    [f'EVT-{i:07d}' for i in range(1, N + 1)],
    'user_id':     rng.integers(1000, 1099, size=N),
    'event_ts':    [
        datetime(2024, 1, 1) + timedelta(hours=int(h))
        for h in rng.integers(0, 720, size=N)
    ],
    'event_type':  rng.choice(['click', 'view', 'purchase', 'login'], size=N),
    'amount':      rng.uniform(1.0, 300.0, size=N).round(2),
    'email':       [f'user{i}@pipeline.example.com' for i in rng.integers(1000, 1099, size=N)],
})

print('Capstone raw data:', raw_pipeline_df.shape)
raw_pipeline_df.head(3)

In [ ]:
# ---- Define two named suites: pipeline.schema and pipeline.values ----

# pipeline.schema — structural integrity
suite_ps = ctx.suites.add(gx.ExpectationSuite(name='pipeline.schema'))
suite_ps.add_expectation(
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=['event_id', 'user_id', 'event_ts', 'event_type', 'amount', 'email'],
        exact_match=True,
    )
)
for col in ['event_id', 'user_id', 'event_ts', 'amount']:
    suite_ps.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col))
suite_ps.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column='event_id'))
suite_ps.add_expectation(gx.expectations.ExpectTableRowCountToBeBetween(min_value=1, max_value=10_000_000))
ctx.suites.save(suite_ps)

# pipeline.values — business-rule content checks
suite_pv = ctx.suites.add(gx.ExpectationSuite(name='pipeline.values'))
suite_pv.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column='amount', min_value=0.01, max_value=10_000.0)
)
suite_pv.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='event_type', value_set=['click', 'view', 'purchase', 'login']
    )
)
suite_pv.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column='email', regex=r'^[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}$'
    )
)
suite_pv.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column='event_id', regex=r'^EVT-\d{7}$'
    )
)
ctx.suites.save(suite_pv)

print('pipeline.schema:', len(suite_ps.expectations), 'expectations')
print('pipeline.values:', len(suite_pv.expectations), 'expectations')

In [ ]:
# ---- Data source + per-stage batch definitions ----

def _make_bd(ctx, name: str):
    """Create a pandas DataSource + asset + whole-dataframe batch def."""
    ds    = ctx.data_sources.add_pandas(f'{name}_src')
    asset = ds.add_dataframe_asset(f'{name}_asset')
    return asset.add_batch_definition_whole_dataframe(f'{name}_bd')


bd_ingest_cap = _make_bd(ctx, 'cap_ingest')
bd_clean_cap  = _make_bd(ctx, 'cap_clean')
bd_agg_cap    = _make_bd(ctx, 'cap_agg')
bd_load_cap   = _make_bd(ctx, 'cap_load')

# Agg output suite (simple: correct columns, positive values)
suite_agg_out = ctx.suites.add(gx.ExpectationSuite(name='pipeline.agg_output'))
suite_agg_out.add_expectation(
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=['user_id', 'total_amount', 'event_count'], exact_match=True
    )
)
suite_agg_out.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column='total_amount', min_value=0.0)
)
ctx.suites.save(suite_agg_out)

# Validation definitions
vd_ingest_schema = ctx.validation_definitions.add(
    gx.ValidationDefinition(name='cap_vd_ingest_schema', data=bd_ingest_cap,
                            suite=ctx.suites.get('pipeline.schema'))
)
vd_ingest_values = ctx.validation_definitions.add(
    gx.ValidationDefinition(name='cap_vd_ingest_values', data=bd_ingest_cap,
                            suite=ctx.suites.get('pipeline.values'))
)
vd_clean_schema  = ctx.validation_definitions.add(
    gx.ValidationDefinition(name='cap_vd_clean_schema', data=bd_clean_cap,
                            suite=ctx.suites.get('pipeline.schema'))
)
vd_agg_out       = ctx.validation_definitions.add(
    gx.ValidationDefinition(name='cap_vd_agg_out', data=bd_agg_cap,
                            suite=ctx.suites.get('pipeline.agg_output'))
)

print('Validation definitions created')

In [ ]:
# ---- Checkpoints with UpdateDataDocsAction ----

update_docs_cap = gx.checkpoint.UpdateDataDocsAction(name='update_docs_cap')

# Ingest checkpoint: run BOTH suites against raw data in one shot
cp_ingest_cap = ctx.checkpoints.add(
    gx.Checkpoint(
        name='cap_ingest_cp',
        validation_definitions=[vd_ingest_schema, vd_ingest_values],
        actions=[update_docs_cap],
        result_format={'result_format': 'COMPLETE'},
    )
)

# Clean checkpoint
cp_clean_cap = ctx.checkpoints.add(
    gx.Checkpoint(
        name='cap_clean_cp',
        validation_definitions=[vd_clean_schema],
        actions=[update_docs_cap],
        result_format={'result_format': 'SUMMARY'},
    )
)

# Aggregate output checkpoint
cp_agg_cap = ctx.checkpoints.add(
    gx.Checkpoint(
        name='cap_agg_cp',
        validation_definitions=[vd_agg_out],
        actions=[update_docs_cap],
        result_format={'result_format': 'SUMMARY'},
    )
)

print('Checkpoints created: cap_ingest_cp, cap_clean_cp, cap_agg_cp')

In [ ]:
# ---- CI gate ----

def ci_gate(checkpoint_result, stage: str, notebook_mode: bool = True) -> None:
    """Fail the CI job if any suite in the Checkpoint failed.

    Args:
        checkpoint_result: CheckpointResult from checkpoint.run().
        stage:             Human-readable stage name for error messages.
        notebook_mode:     Raise RuntimeError instead of sys.exit(1) in notebooks.
    """
    if not checkpoint_result.success:
        failed_suites = [
            identifier.expectation_suite_identifier.name
            for identifier, vr in checkpoint_result.run_results.items()
            if not vr.success
        ]
        msg = f'[{stage}] CI gate FAILED — suites: {failed_suites}'
        if notebook_mode:
            raise RuntimeError(msg)
        print(msg, file=sys.stderr)
        sys.exit(1)
    print(f'[{stage}] CI gate PASSED')


print('ci_gate() defined')

In [ ]:
# ---- Full capstone pipeline ----

def capstone_pipeline(raw: pd.DataFrame, notebook_mode: bool = True) -> pd.DataFrame:
    """Four-step pipeline: ingest → clean → aggregate → load.

    Each stage validates its input before transforming.
    Raises RuntimeError (notebook_mode=True) or calls sys.exit(1) on failure.
    """
    print('=' * 55)
    print('CAPSTONE PIPELINE RUN')
    print('=' * 55)

    # ---------- INGEST ----------
    print('\n[1/4] INGEST — validating raw input')
    r = cp_ingest_cap.run(batch_parameters={'dataframe': raw})
    ci_gate(r, 'ingest', notebook_mode)
    print(f'      rows: {len(raw)}, cols: {list(raw.columns)}')

    # ---------- CLEAN ----------
    print('\n[2/4] CLEAN — validating after dropping nulls')
    clean = raw.dropna(subset=['event_id', 'user_id', 'event_ts', 'amount']).copy()
    clean['amount'] = clean['amount'].astype(float)
    # Re-validate cleaned frame against schema suite (batch def reuses bd_clean_cap)
    r_clean = cp_clean_cap.run(batch_parameters={'dataframe': clean})
    ci_gate(r_clean, 'clean', notebook_mode)
    print(f'      rows after clean: {len(clean)}')

    # ---------- AGGREGATE ----------
    print('\n[3/4] AGGREGATE — validating aggregated output')
    agg = (
        clean.groupby('user_id', as_index=False)
        .agg(total_amount=('amount', 'sum'), event_count=('event_id', 'count'))
    )
    r_agg = cp_agg_cap.run(batch_parameters={'dataframe': agg})
    ci_gate(r_agg, 'aggregate', notebook_mode)
    print(f'      rows after agg: {len(agg)}')

    # ---------- LOAD ----------
    print('\n[4/4] LOAD — writing to destination (simulated)')
    loaded = agg.sort_values('total_amount', ascending=False).reset_index(drop=True)
    # (In production: write to database / file / API here)
    print(f'      rows loaded: {len(loaded)}')

    print('\n' + '=' * 55)
    print('PIPELINE COMPLETE')
    print('=' * 55)
    return loaded


# Run on clean data
output = capstone_pipeline(raw_pipeline_df)
print()
print('Top 5 users by total amount:')
print(output.head())

In [ ]:
# ---- Confirm CI gate fires on bad row injection ----

print('Injecting bad rows to confirm CI gate fires...')
print()

bad_rows = pd.DataFrame([
    {
        'event_id':   'EVT-0000001',   # DUPLICATE — violates uniqueness
        'user_id':    1001,
        'event_ts':   datetime(2024, 1, 15),
        'event_type': 'UNKNOWN',        # not in value_set — violates pipeline.values
        'amount':     -99.99,           # negative — violates pipeline.values
        'email':      'not-an-email',   # invalid format — violates pipeline.values
    },
])

dirty_pipeline_df = pd.concat([raw_pipeline_df, bad_rows], ignore_index=True)
print(f'Dirty DataFrame: {len(dirty_pipeline_df)} rows (+1 bad row)')

try:
    capstone_pipeline(dirty_pipeline_df, notebook_mode=True)
except RuntimeError as exc:
    print(f'\nCI gate fired as expected:')
    print(f'  {exc}')
    print()
    print('The pipeline was aborted before the CLEAN stage ran.')
    print('Data Docs HTML report was generated with failure details.')

## README: Suite Design Decisions, Drift Strategy, and Extension Guide

### Suite Design Decisions

**Two suites instead of one.** Splitting into `pipeline.schema` and `pipeline.values` lets operators respond differently to each failure type:
- Schema failures (missing columns, null critical fields, duplicate IDs) are engineering bugs — they indicate an upstream system change. Page the on-call engineer.
- Value failures (bad email formats, out-of-range amounts, unknown event types) are data quality issues — they may be handled by quarantine or enrichment without halting the pipeline.

**`exact_match=True` on `ExpectTableColumnsToMatchSet`.** This is the most important schema-drift guard. Without it, upstream systems can silently add columns that later cause issues in downstream SELECT statements or ML feature pipelines.

**`ExpectColumnValuesToBeUnique` on the primary key.** Duplicate event IDs are a silent killer — they inflate metrics and cause double-counting in aggregates. Catch them at ingest.

### Schema Drift Strategy

1. Lock the ingest column set with `ExpectTableColumnsToMatchSet(exact_match=True)`.
2. Run `drift_report(df, suite)` on any failure to print added/removed columns.
3. Treat any schema change as a breaking change: update the suite, tag the previous version (see `version_suite()` from Day 4), commit both to version control.
4. For intentional schema evolution, use a migration playbook: update the upstream producer, then update the suite in the same PR so the gate never breaks in CI.

### How to Add a New Expectation

**Built-in expectation:** Instantiate it from `gx.expectations.*` and call `suite.add_expectation()`.

**Custom expectation:** Subclass `ColumnMapExpectation` (or `TableExpectation` for table-level checks):

```python
class ExpectColumnValuesToBeValidISBN(gx.expectations.ColumnMapExpectation):
    map_metric = 'column_values.match_isbn'
    success_keys = ('mostly',)

    def _validate(self, column, mostly=1.0, **kwargs):
        isbn_re = re.compile(r'^(?:\d{9}[\dX]|\d{13})$')
        valid = column.astype(str).map(lambda v: bool(isbn_re.match(v)))
        unexpected = column[~valid].tolist()
        success = (1 - len(unexpected) / len(column)) >= mostly
        return {'success': success, 'result': {'unexpected_list': unexpected}}

    @classmethod
    def _prescriptive_renderer(cls, configuration=None, **kwargs):
        col = configuration.kwargs.get('column', '?') if configuration else '?'
        return f'All values in "{col}" must be valid ISBN-10 or ISBN-13 identifiers.'
```

Always test:
1. That `unexpected_list` contains exactly the known-bad values.
2. That `_prescriptive_renderer` returns a non-empty string.
3. That running a Checkpoint with `UpdateDataDocsAction` produces readable HTML (not raw JSON) for the expectation.

## Challenge — Extend Your GX Validation Layer

Choose **two** of the following extensions to complete before the Recap:

**Challenge A — Second Custom Expectation**
Implement  by subclassing .
The expectation should validate that every value in a column matches the  format.
Include a  so it displays cleanly in Data Docs.
Test it on a DataFrame with a mix of valid ISO dates and invalid strings.

**Challenge B — Fifth Pipeline Stage**
Add a  stage to the four-step pipeline that reads the aggregated output
and validates it with a third suite .
The suite should include: , 
, and .
Run the full five-stage pipeline and confirm all Checkpoints pass.

**Challenge C — Slack Notification Action**
Add a  to your multi-suite Checkpoint (mock the Slack webhook URL).
Inject a bad row so the Checkpoint fails, and print the payload that would have been sent
to Slack (the  summary as a dict).
Hint: you can subclass  to create a  that
prints instead of POSTing, making this runnable without a real Slack workspace.


## Course Recap — Days 4–7

| Day | Topic | Key skill |
|---|---|---|
| 4 | Expectation Suites | Create, version, and round-trip suite JSON |
| 5 | Checkpoints and Data Docs | Orchestrate multi-suite validation; CI gate pattern |
| 6 | Pipeline Integration | Validate inputs at each stage; detect schema drift |
| 7 | Custom Expectations + Capstone | Subclass `ColumnMapExpectation`; full pipeline end-to-end |

**The core mental model:**

```
DataContract (Suite JSON)
    ↓  validated by
Checkpoint (multi-suite)
    ↓  produces
CheckpointResult
    ↓  inspected by
CI Gate  →  sys.exit(1) on failure
    ↓  documented by
Data Docs HTML
```

Congratulations on completing the Great Expectations for Data Quality course.